In [1]:
from pathlib import Path
import pandas as pd

project_root = Path("/home/dwp46550/ba_nylon")

df = pd.read_csv(
    project_root / "matrices" / "NylC_Puetz_raw_data.CSV",
    sep=";",
    decimal=","
)
print(df.shape)

(36, 7)


### Build mutation-based sequence features for experimentally characterized variants

This cell loads the wild-type NylC reference sequence and all known variant FASTA files. Each variant sequence is compared against the wild-type sequence to extract amino acid substitutions in standard mutation notation, for example `D304M` or `F134W`.

For every variant, the code computes mutation-level descriptors such as the number of mutations, changes in charge, hydrophobicity, and side-chain volume. It also classifies mutated positions into literature-supported positions, MD-contact positions, and unknown positions. Finally, mutation identities and mutated positions are one-hot encoded so that the variants can be used as input for machine learning models.

In [ ]:
from pathlib import Path
import re
import pandas as pd
from Bio import SeqIO
from sklearn.preprocessing import MultiLabelBinarizer
# Define project paths
project_root = Path("/home/dwp46550/ba_nylon")
# Load the reference FASTA sequence
known_fasta_dir = project_root / "inputs" / "variant_fastas"
reference_fasta = known_fasta_dir / "WT.fasta"


def read_fasta_sequence(path):
    record = next(SeqIO.parse(path, "fasta"))
    return str(record.seq)

reference_seq = read_fasta_sequence(reference_fasta)
# Amino acid physicochemical property table
aa_properties = {
    "A": {"charge": 0, "hydrophobicity": 1.8, "volume": 88.6},
    "R": {"charge": 1, "hydrophobicity": -4.5, "volume": 173.4},
    "N": {"charge": 0, "hydrophobicity": -3.5, "volume": 114.1},
    "D": {"charge": -1, "hydrophobicity": -3.5, "volume": 111.1},
    "C": {"charge": 0, "hydrophobicity": 2.5, "volume": 108.5},
    "Q": {"charge": 0, "hydrophobicity": -3.5, "volume": 143.8},
    "E": {"charge": -1, "hydrophobicity": -3.5, "volume": 138.4},
    "G": {"charge": 0, "hydrophobicity": -0.4, "volume": 60.1},
    "H": {"charge": 0.5, "hydrophobicity": -3.2, "volume": 153.2},
    "I": {"charge": 0, "hydrophobicity": 4.5, "volume": 166.7},
    "L": {"charge": 0, "hydrophobicity": 3.8, "volume": 166.7},
    "K": {"charge": 1, "hydrophobicity": -3.9, "volume": 168.6},
    "M": {"charge": 0, "hydrophobicity": 1.9, "volume": 162.9},
    "F": {"charge": 0, "hydrophobicity": 2.8, "volume": 189.9},
    "P": {"charge": 0, "hydrophobicity": -1.6, "volume": 112.7},
    "S": {"charge": 0, "hydrophobicity": -0.8, "volume": 89.0},
    "T": {"charge": 0, "hydrophobicity": -0.7, "volume": 116.1},
    "W": {"charge": 0, "hydrophobicity": -0.9, "volume": 227.8},
    "Y": {"charge": 0, "hydrophobicity": -1.3, "volume": 193.6},
    "V": {"charge": 0, "hydrophobicity": 4.2, "volume": 140.0},
}
# Positions with prior experimental or literature support Puetz et all
known_positions = {99, 134, 301, 304, 330}

# Positions identified as substrate-contact residues in MD analysis
md_contact_positions = {91, 98, 111, 137, 139, 144, 146, 305}

# Extract all substitutions relative to the reference sequence
def extract_mutations(reference_seq, variant_seq):
    if len(reference_seq) != len(variant_seq):
        raise ValueError(
            f"Sequence length mismatch: reference={len(reference_seq)}, variant={len(variant_seq)}"
        )

    mutations = []
    # Parse the residue number from a mutation string
    for pos, (wt_aa, var_aa) in enumerate(zip(reference_seq, variant_seq), start=1):
        if wt_aa != var_aa:
            mutations.append(f"{wt_aa}{pos}{var_aa}")

    return mutations

def mutation_position(mutation):
    return int(re.findall(r"\d+", mutation)[0])

# Build a feature table from all FASTA variants
def build_sequence_feature_table(fasta_dir, reference_seq):
    rows = []

    for fasta_file in sorted(Path(fasta_dir).glob("*.fasta")):
        variant = fasta_file.stem
        seq = read_fasta_sequence(fasta_file)
        mutations = extract_mutations(reference_seq, seq)

        delta_charge = 0
        delta_hydrophobicity = 0
        delta_volume = 0

        n_known_position_mut = 0
        n_md_contact_mut = 0
        n_unknown_position_mut = 0
        # Calculate physicochemical mutation descriptors
        for mut in mutations:
            wt_aa = mut[0]
            new_aa = mut[-1]
            pos = mutation_position(mut)

            delta_charge += aa_properties[new_aa]["charge"] - aa_properties[wt_aa]["charge"]
            delta_hydrophobicity += aa_properties[new_aa]["hydrophobicity"] - aa_properties[wt_aa]["hydrophobicity"]
            delta_volume += aa_properties[new_aa]["volume"] - aa_properties[wt_aa]["volume"]

            # Classify mutations by positional evidence
            #Add binary features for key substitutions and epistatic patterns
            if pos in known_positions:
                n_known_position_mut += 1
            elif pos in md_contact_positions:
                n_md_contact_mut += 1
            else:
                n_unknown_position_mut += 1

        rows.append({
            "variant": variant,
            "sequence": seq,
            "mutations": ";".join(mutations),
            "mutation_list": mutations,
            "position_list": [str(mutation_position(m)) for m in mutations],
            "n_mut": len(mutations),
            "delta_charge": delta_charge,
            "delta_hydrophobicity": delta_hydrophobicity,
            "delta_volume": delta_volume,
            "n_known_position_mut": n_known_position_mut,
            "n_md_contact_mut": n_md_contact_mut,
            "n_unknown_position_mut": n_unknown_position_mut,
            "has_D99R": int("D99R" in mutations),
            "has_F134W": int("F134W" in mutations),
            "has_D304M": int("D304M" in mutations),
            "has_R330A": int("R330A" in mutations),
            "epistasis_D99R_D304": int(
                ("D99R" in mutations) and any(m.startswith("D304") for m in mutations)
            ),
            "hp_like_core": int(
                ("F134W" in mutations) and ("D304M" in mutations) and ("R330A" in mutations)
            ),
        })

    base_df = pd.DataFrame(rows)

    mlb_mut = MultiLabelBinarizer()
    mut_onehot = pd.DataFrame(
        mlb_mut.fit_transform(base_df["mutation_list"]),
        columns=[f"mut_{m}" for m in mlb_mut.classes_],
        index=base_df.index
    )

    mlb_pos = MultiLabelBinarizer()
    pos_onehot = pd.DataFrame(
        mlb_pos.fit_transform(base_df["position_list"]),
        columns=[f"pos_{p}" for p in mlb_pos.classes_],
        index=base_df.index
    )

    feature_df = pd.concat(
        [
            base_df.drop(columns=["sequence", "mutation_list", "position_list"]),
            mut_onehot,
            pos_onehot,
        ],
        axis=1
    )

    return feature_df

known_feature_df = build_sequence_feature_table(known_fasta_dir, reference_seq)

display(known_feature_df.head())
print(known_feature_df.shape)

,variant,mutations,n_mut,delta_charge,delta_hydrophobicity,delta_volume,n_known_position_mut,n_md_contact_mut,n_unknown_position_mut,has_D99R,...,mut_D99V,mut_F134W,mut_F301L,mut_R330A,mut_R330Q,pos_134,pos_301,pos_304,pos_330,pos_99
0,D304E,D304E,1,0,0.0,27.3,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,D304L,D304L,1,1,7.3,55.6,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,D304M,D304M,1,1,5.4,51.8,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,D304Q,D304Q,1,1,0.0,32.7,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,D304R,D304R,1,2,-1.0,62.3,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0


(35, 36)


### Train activity and thermostability prediction models

This cell merges the sequence-derived feature table with experimental labels for PA6 activity and melting temperature. The data are cleaned by converting activity and melting temperature values into numeric format.

The model uses numeric mutation features as predictors and trains two separate ElasticNet regression pipelines: one for enzymatic activity and one for thermostability. Both models use standardization followed by ElasticNetCV. Leave-one-out cross-validation is used because the experimental dataset is small. The fitted models are later used to predict activity and thermostability for BoltzGen-generated variants.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np
import pandas as pd

# Prepare experimental lab labels
lab_data = df.copy()

# Rename columns to consistent model input names
lab_data = lab_data.rename(columns={
    "variant_id": "variant",
    "activity_pa6": "activity",
    "tm_celsius": "tm",
})

# Convert activity values to numeric format
lab_data["activity"] = pd.to_numeric(
    lab_data["activity"],
    errors="coerce"
)
# Convert melting temperature values to numeric format
lab_data["tm"] = (
    lab_data["tm"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

lab_data["tm"] = pd.to_numeric(
    lab_data["tm"],
    errors="coerce"
)

#Merge sequence features with experimental labels
model_df = known_feature_df.merge(
    lab_data[["variant", "activity", "tm"]],
    on="variant",
    how="inner"
)

print("Merged shape:", model_df.shape)
display(model_df.head())

# Select numeric feature columns for modeling
non_feature_cols = {
    "variant",
    "mutations",
    "sequence",
    "activity",
    "tm",
}

feature_cols = [
    c for c in model_df.columns
    if c not in non_feature_cols
    and pd.api.types.is_numeric_dtype(model_df[c])
]

# Optional: small, safer feature set for n=36
# Uncomment this if the full feature set gives unstable results.
# feature_cols = [
#     "n_mut",
#     "delta_charge",
#     "delta_hydrophobicity",
#     "delta_volume",
#     "n_known_position_mut",
#     "n_md_contact_mut",
#     "n_unknown_position_mut",
#     "has_D99R",
#     "has_F134W",
#     "has_D304M",
#     "has_R330A",
#     "epistasis_D99R_D304",
#     "hp_like_core",
# ]

print("Number of features:", len(feature_cols))
print(feature_cols)

# Train the PA6 activity prediction model
activity_model_df = model_df.dropna(subset=["activity"]).copy()

X_activity = activity_model_df[feature_cols].fillna(0)
y_activity = activity_model_df["activity"]

# Evaluate activity prediction by leave-one-out cross-validation
loo_activity = LeaveOneOut()

activity_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
        alphas=np.logspace(-3, 2, 100),
        cv=loo_activity,
        max_iter=100000,
        random_state=42
    ))
])

# Fit the final activity model on all labeled activity data
activity_pred_loo = cross_val_predict(
    activity_model,
    X_activity,
    y_activity,
    cv=loo_activity
)

print("Activity LOOCV")
print("R2:", r2_score(y_activity, activity_pred_loo))
print("MAE:", mean_absolute_error(y_activity, activity_pred_loo))

activity_model.fit(X_activity, y_activity)

# Train the melting temperature prediction model
tm_model_df = model_df.dropna(subset=["tm"]).copy()

X_tm = tm_model_df[feature_cols].fillna(0)
y_tm = tm_model_df["tm"]

# Evaluate melting temperature prediction by leave-one-out cross-validation
loo_tm = LeaveOneOut()

# Fit the final thermostability model on all labeled Tm data
tm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
        alphas=np.logspace(-3, 2, 100),
        cv=loo_tm,
        max_iter=100000,
        random_state=42
    ))
])

tm_pred_loo = cross_val_predict(
    tm_model,
    X_tm,
    y_tm,
    cv=loo_tm
)

print("\nTm LOOCV")
print("R2:", r2_score(y_tm, tm_pred_loo))
print("MAE:", mean_absolute_error(y_tm, tm_pred_loo))

tm_model.fit(X_tm, y_tm)

Merged shape: (35, 38)


,variant,mutations,n_mut,delta_charge,delta_hydrophobicity,delta_volume,n_known_position_mut,n_md_contact_mut,n_unknown_position_mut,has_D99R,...,mut_F301L,mut_R330A,mut_R330Q,pos_134,pos_301,pos_304,pos_330,pos_99,activity,tm
0,D304E,D304E,1,0,0.0,27.3,1,0,0,0,...,0,0,0,0,0,1,0,0,171,87.9
1,D304L,D304L,1,1,7.3,55.6,1,0,0,0,...,0,0,0,0,0,1,0,0,138,89.6
2,D304M,D304M,1,1,5.4,51.8,1,0,0,0,...,0,0,0,0,0,1,0,0,234,89.5
3,D304Q,D304Q,1,1,0.0,32.7,1,0,0,0,...,0,0,0,0,0,1,0,0,157,89.6
4,D304R,D304R,1,2,-1.0,62.3,1,0,0,0,...,0,0,0,0,0,1,0,0,123,88.4


Number of features: 34
['n_mut', 'delta_charge', 'delta_hydrophobicity', 'delta_volume', 'n_known_position_mut', 'n_md_contact_mut', 'n_unknown_position_mut', 'has_D99R', 'has_F134W', 'has_D304M', 'has_R330A', 'epistasis_D99R_D304', 'hp_like_core', 'mut_D304E', 'mut_D304I', 'mut_D304L', 'mut_D304M', 'mut_D304Q', 'mut_D304R', 'mut_D304S', 'mut_D304V', 'mut_D304W', 'mut_D99G', 'mut_D99R', 'mut_D99V', 'mut_F134W', 'mut_F301L', 'mut_R330A', 'mut_R330Q', 'pos_134', 'pos_301', 'pos_304', 'pos_330', 'pos_99']
Activity LOOCV
R2: 0.4490380988559425
MAE: 59.82083196263575

Tm LOOCV
R2: 0.4760134065101629
MAE: 1.3788637422735577


Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 ElasticNetCV(alphas=array([1.00000000e-03, 1.12332403e-03, 1.26185688e-03, 1.41747416e-03,
       1.59228279e-03, 1.78864953e-03, 2.00923300e-03, 2.25701972e-03,
       2.53536449e-03, 2.84803587e-03, 3.19926714e-03, 3.59381366e-03,
       4.03701726e-03, 4.53487851e-03, 5.09413801e-03, 5.72236766e-03,
       6.42807312e-03, 7.22080902e-03,...
       1.09749877e+01, 1.23284674e+01, 1.38488637e+01, 1.55567614e+01,
       1.74752840e+01, 1.96304065e+01, 2.20513074e+01, 2.47707636e+01,
       2.78255940e+01, 3.12571585e+01, 3.51119173e+01, 3.94420606e+01,
       4.43062146e+01, 4.97702356e+01, 5.59081018e+01, 6.28029144e+01,
       7.05480231e+01, 7.92482898e+01, 8.90215085e+01, 1.00000000e+02]),
                              cv=LeaveOneOut(),
                              l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
                              max_iter=100000, random_state=42))])

### Read BoltzGen design tables and map generated variants to the wild-type sequence

This cell reads all BoltzGen result tables from the `boltz_variants` directory. Each run is annotated with a run name derived from the CSV filename, and each design receives a unique `variant_id`.

Because the BoltzGen output contains partial designed chain sequences rather than full-length NylC sequences, each designed chain is locally aligned to the wild-type sequence using Biopython's `PairwiseAligner`. The alignment is used to determine the corresponding wild-type residue interval and to extract all amino acid substitutions. The resulting table combines all BoltzGen designs across runs into a single DataFrame.

In [30]:
from pathlib import Path
import pandas as pd
from Bio.Align import PairwiseAligner
# Define input paths
project_root = Path("/home/dwp46550/ba_nylon")

boltz_dir = project_root / "inputs" / "boltz_variants"
wt_fasta = project_root / "inputs" / "variant_fastas" / "WT.fasta"

# Read all BoltzGen result CSV files
csv_files = sorted(boltz_dir.glob("all_designs_metrics_*.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in {boltz_dir}")

print("Found CSV files:")
for f in csv_files:
    print(f.name)

# Load the wild-type FASTA sequence
with open(wt_fasta) as f:
    wt_seq = "".join(
        line.strip()
        for line in f
        if not line.startswith(">")
    )

print(f"\nWT length: {len(wt_seq)}")

# Configure a local pairwise sequence aligner
aligner = PairwiseAligner()
aligner.mode = "local"
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

# Align each designed chain sequence to the wild-type sequence
def align_and_compare(design_seq, wt_seq):
    design_seq_raw = str(design_seq)
    design_seq_clean = design_seq_raw.replace("X", "")
    # Remove unknown residues before alignment
    if design_seq_clean == "" or design_seq_clean.lower() == "nan":
        return pd.Series({
            "wt_start": None,
            "wt_end": None,
            "alignment_score": None,
            "designed_chain_length_raw": len(design_seq_raw),
            "designed_chain_length_clean": None,
            "n_mutations": None,
            "mutations": None
        })

    alignment = aligner.align(wt_seq, design_seq_clean)[0]
    # Extract aligned wild-type and design sequence blocks
    wt_blocks = alignment.aligned[0]
    design_blocks = alignment.aligned[1]

    mutations = []
    # Convert residue differences into mutation strings
    for (wt_start, wt_end), (des_start, des_end) in zip(wt_blocks, design_blocks):
        wt_fragment = wt_seq[wt_start:wt_end]
        design_fragment = design_seq_clean[des_start:des_end]

        for offset, (wt_aa, design_aa) in enumerate(zip(wt_fragment, design_fragment)):
            wt_pos = wt_start + offset + 1

            if wt_aa != design_aa:
                mutations.append(f"{wt_aa}{wt_pos}{design_aa}")

    return pd.Series({
        "wt_start": int(wt_blocks[0][0] + 1),
        "wt_end": int(wt_blocks[-1][1]),
        "alignment_score": float(alignment.score),
        "designed_chain_length_raw": len(design_seq_raw),
        "designed_chain_length_clean": len(design_seq_clean),
        "n_mutations": len(mutations),
        "mutations": ",".join(mutations)
    })


all_runs = []
# Read each BoltzGen run and annotate it with run metadata
for csv_file in csv_files:
    run_name = csv_file.stem.replace("all_designs_metrics_", "")

    df_run = pd.read_csv(csv_file)
    df_run["boltzgen_run"] = run_name
    df_run["source_csv"] = csv_file.name

    if "designed_chain_sequence" not in df_run.columns:
        raise KeyError(
            f"'designed_chain_sequence' not found in {csv_file.name}. "
            f"Available columns: {df_run.columns.tolist()}"
        )

    comparison = df_run["designed_chain_sequence"].apply(
        lambda seq: align_and_compare(seq, wt_seq)
    )

    df_run_aligned = pd.concat([df_run, comparison], axis=1)
    all_runs.append(df_run_aligned)

df_all_boltzgen = pd.concat(all_runs, ignore_index=True)

# Add a unique variant identifier across all runs
df_all_boltzgen["variant_id"] = (
    df_all_boltzgen["boltzgen_run"].astype(str)
    + "__"
    + df_all_boltzgen["id"].astype(str)
)

# Keep all BoltzGen columns and append alignment-derived mutation information
model_input_cols = [
    "variant_id",
    "boltzgen_run",
    "source_csv",
    "id",
    "final_rank",
    "designed_sequence",
    "designed_chain_sequence",
    "wt_start",
    "wt_end",
    "alignment_score",
    "designed_chain_length_raw",
    "designed_chain_length_clean",
    "n_mutations",
    "mutations",
]

model_input_cols = [
    c for c in model_input_cols
    if c in df_all_boltzgen.columns
]

df_boltzgen_variants = df_all_boltzgen.copy()

print("\nCombined BoltzGen variants:")
print(df_boltzgen_variants.shape)

preview_cols = [
    "variant_id",
    "boltzgen_run",
    "source_csv",
    "id",
    "final_rank",
    "wt_start",
    "wt_end",
    "n_mutations",
    "mutations",
]

preview_cols = [
    c for c in preview_cols
    if c in df_boltzgen_variants.columns
]

print(
    df_boltzgen_variants[preview_cols]
    .sort_values(["boltzgen_run", "final_rank"])
    .head(30)
)
# Normalize the rank within each BoltzGen run
df_boltzgen_variants["run_rank_norm"] = (
    df_boltzgen_variants["final_rank"]
    / df_boltzgen_variants.groupby("boltzgen_run")["final_rank"].transform("max")
)

# Extract mutated residue positions
df_boltzgen_variants["mutation_positions"] = (
    df_boltzgen_variants["mutations"]
    .str.findall(r"\d+")
    .apply(lambda x: [int(i) for i in x] if isinstance(x, list) else [])
)
# Store the mutation count
df_boltzgen_variants["mutation_count"] = (
    df_boltzgen_variants["mutations"]
    .str.count(",")
    .fillna(0)
    + 1
)

#print(df_boltzgen_variants.head(20))
#print(df_boltzgen_variants.keys())

# Optional save file
# out_file = boltz_dir / "combined_boltzgen_variants_aligned_to_wt.csv"
# df_boltzgen_variants.to_csv(out_file, index=False)

#print(f"\nSaved combined table to:\n{out_file}")

Found CSV files:
all_designs_metrics_HP_pocket.csv
all_designs_metrics_NylC_double.csv
all_designs_metrics_NylC_negativ.csv
all_designs_metrics_NylC_paper.csv
all_designs_metrics_NylC_pocket.csv

WT length: 355

Combined BoltzGen variants:
(158, 268)
                     variant_id boltzgen_run  \
0       HP_pocket__HP_pocket_09    HP_pocket   
1       HP_pocket__HP_pocket_10    HP_pocket   
2       HP_pocket__HP_pocket_02    HP_pocket   
3       HP_pocket__HP_pocket_14    HP_pocket   
4       HP_pocket__HP_pocket_05    HP_pocket   
5       HP_pocket__HP_pocket_07    HP_pocket   
6       HP_pocket__HP_pocket_06    HP_pocket   
7       HP_pocket__HP_pocket_04    HP_pocket   
8       HP_pocket__HP_pocket_08    HP_pocket   
9       HP_pocket__HP_pocket_00    HP_pocket   
10      HP_pocket__HP_pocket_01    HP_pocket   
11      HP_pocket__HP_pocket_03    HP_pocket   
12      HP_pocket__HP_pocket_23    HP_pocket   
13      HP_pocket__HP_pocket_15    HP_pocket   
14      HP_pocket__HP_pocket_

### Convert BoltzGen mutation lists into the same feature space as the training variants

This cell transforms the mutation lists extracted from the BoltzGen alignments into the same feature representation that was used for the experimentally characterized training variants. It computes mutation counts, physicochemical changes, positional classes, key mutation indicators, and one-hot encoded mutation and position features.

This step is necessary because the trained activity and thermostability models expect the exact same feature columns as the original training dataset.

In [ ]:
import re
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

# Build model-compatible features directly from BoltzGen mutation strings
def build_feature_table_from_mutations(df_variants):

    rows = []

    for _, row in df_variants.iterrows():

        mutations = []
        #Parse mutation strings from the alignment-derived mutation column
        if pd.notna(row["mutations"]) and row["mutations"] != "":
            mutations = [
                m.strip()
                for m in str(row["mutations"]).split(",")
            ]
        # Compute physicochemical changes for each substitution
        delta_charge = 0
        delta_hydrophobicity = 0
        delta_volume = 0

        n_known_position_mut = 0
        n_md_contact_mut = 0
        n_unknown_position_mut = 0

        for mut in mutations:

            wt_aa = mut[0]
            new_aa = mut[-1]
            pos = int(re.findall(r"\d+", mut)[0])

            delta_charge += (
                aa_properties[new_aa]["charge"]
                - aa_properties[wt_aa]["charge"]
            )

            delta_hydrophobicity += (
                aa_properties[new_aa]["hydrophobicity"]
                - aa_properties[wt_aa]["hydrophobicity"]
            )

            delta_volume += (
                aa_properties[new_aa]["volume"]
                - aa_properties[wt_aa]["volume"]
            )
            # Classify positions as known, MD-contact, or unknown
            if pos in known_positions:
                n_known_position_mut += 1
            elif pos in md_contact_positions:
                n_md_contact_mut += 1
            else:
                n_unknown_position_mut += 1

        rows.append({
            "variant": row["variant_id"],
            "mutations": ";".join(mutations),
            "mutation_list": mutations,
            "position_list": [
                str(int(re.findall(r'\d+', m)[0]))
                for m in mutations
            ],
            "n_mut": len(mutations),
            "delta_charge": delta_charge,
            "delta_hydrophobicity": delta_hydrophobicity,
            "delta_volume": delta_volume,
            "n_known_position_mut": n_known_position_mut,
            "n_md_contact_mut": n_md_contact_mut,
            "n_unknown_position_mut": n_unknown_position_mut,
            "has_D99R": int("D99R" in mutations),
            "has_F134W": int("F134W" in mutations),
            "has_D304M": int("D304M" in mutations),
            "has_R330A": int("R330A" in mutations),
            "epistasis_D99R_D304": int(
                ("D99R" in mutations)
                and any(m.startswith("D304") for m in mutations)
            ),
            "hp_like_core": int(
                ("F134W" in mutations)
                and ("D304M" in mutations)
                and ("R330A" in mutations)
            ),
        })

    base_df = pd.DataFrame(rows)

    mlb_mut = MultiLabelBinarizer()
    mut_onehot = pd.DataFrame(
        mlb_mut.fit_transform(base_df["mutation_list"]),
        columns=[f"mut_{m}" for m in mlb_mut.classes_],
        index=base_df.index
    )

    mlb_pos = MultiLabelBinarizer()
    pos_onehot = pd.DataFrame(
        mlb_pos.fit_transform(base_df["position_list"]),
        columns=[f"pos_{p}" for p in mlb_pos.classes_],
        index=base_df.index
    )

    feature_df = pd.concat(
        [
            base_df.drop(
                columns=["mutation_list", "position_list"]
            ),
            mut_onehot,
            pos_onehot,
        ],
        axis=1
    )
    # Return a model-compatible feature table
    return feature_df

### Predict PA6 activity and thermostability for BoltzGen-generated variants

This cell applies the trained ElasticNet models to the BoltzGen variants. The BoltzGen feature table is aligned to the same feature columns used during model training. Missing one-hot encoded features are added as zeros, which indicates that the corresponding mutation or position is absent in a given BoltzGen variant.

The activity and melting temperature predictions are merged back into the main BoltzGen variant table for downstream ranking and candidate selection.

In [33]:
# -----------------------------
# Predict activity and Tm for BoltzGen variants
# -----------------------------

boltz_feature_df = build_feature_table_from_mutations(
    df_boltzgen_variants
)

missing_features = [
    c for c in feature_cols
    if c not in boltz_feature_df.columns
]

extra_features = [
    c for c in boltz_feature_df.columns
    if c not in feature_cols
    and c not in ["variant", "mutations"]
]

print("BoltzGen feature shape:", boltz_feature_df.shape)
print("Missing features:", missing_features)
print("Extra features:", extra_features)

for col in missing_features:
    boltz_feature_df[col] = 0

X_boltz = boltz_feature_df[feature_cols].fillna(0)

df_predictions = boltz_feature_df[["variant"]].copy()
df_predictions["predicted_activity"] = activity_model.predict(X_boltz)
df_predictions["predicted_tm"] = tm_model.predict(X_boltz)

# Alte Predictions entfernen, falls Zelle mehrfach ausgeführt wurde
df_boltzgen_variants = df_boltzgen_variants.drop(
    columns=[
        "predicted_activity",
        "predicted_tm",
        "variant",
    ],
    errors="ignore"
)

df_boltzgen_variants = df_boltzgen_variants.merge(
    df_predictions,
    left_on="variant_id",
    right_on="variant",
    how="left"
).drop(columns=["variant"])

print(
    df_boltzgen_variants[
        [
            "variant_id",
            "boltzgen_run",
            "final_rank",
            "n_mutations",
            "mutations",
            "predicted_activity",
            "predicted_tm",
        ]
    ]
    .sort_values("predicted_activity", ascending=False)
    .head(20)
)

BoltzGen feature shape: (158, 58)
Missing features: ['mut_D304E', 'mut_D304I', 'mut_D304L', 'mut_D304M', 'mut_D304Q', 'mut_D304R', 'mut_D304S', 'mut_D304V', 'mut_D304W', 'mut_D99V', 'mut_F301L', 'mut_R330A', 'mut_R330Q', 'pos_301', 'pos_304', 'pos_330']
Extra features: ['mut_A91G', 'mut_A91R', 'mut_D99A', 'mut_D99L', 'mut_D99N', 'mut_D99S', 'mut_D99T', 'mut_F134A', 'mut_F134I', 'mut_F134L', 'mut_F134P', 'mut_F134V', 'mut_G111F', 'mut_G111V', 'mut_G111Y', 'mut_L137A', 'mut_L137E', 'mut_L139F', 'mut_L139G', 'mut_L139I', 'mut_L139R', 'mut_L139V', 'mut_L139Y', 'mut_V144A', 'mut_V144G', 'mut_V144S', 'mut_V144T', 'mut_Y146H', 'mut_Y146N', 'mut_Y98G', 'mut_Y98S', 'pos_111', 'pos_137', 'pos_139', 'pos_144', 'pos_146', 'pos_91', 'pos_98']
                 variant_id boltzgen_run  final_rank  n_mutations  \
6   HP_pocket__HP_pocket_06    HP_pocket           7            7   
7   HP_pocket__HP_pocket_04    HP_pocket           8            8   
10  HP_pocket__HP_pocket_01    HP_pocket          11 

### Select a balanced set of 20 variants for experimental testing

This cell ranks BoltzGen-generated variants by combining model predictions, structure-derived BoltzGen metrics, predicted thermostability, and mutation novelty. Instead of selecting only the top predicted activity variants, the final list is divided into three groups.

Group A contains the top 10 variants by predicted activity and represents exploitation of the model. Group B contains 5 variants with unusual mutation patterns and acceptable structure scores to explore less familiar design space. Group C contains 5 variants with strong structure scores but only intermediate model predictions, serving as a structural-control group.

This selection strategy is designed to balance the probability of finding improved variants with the scientific value of testing diverse and model-challenging candidates.

In [34]:
import numpy as np
import pandas as pd
import re

df = df_boltzgen_variants.copy()

model_col = "predicted_activity"
tm_col = "predicted_tm"

if model_col not in df.columns:
    raise KeyError(f"{model_col} not found.")

if tm_col not in df.columns:
    raise KeyError(f"{tm_col} not found.")

structure_cols_high_good = [
    "design_to_target_iptm",
    "design_ptm",
    "plip_hbonds_refolded",
    "delta_sasa_refolded",
]

structure_cols_low_good = [
    "filter_rmsd",
    "designfolding-filter_rmsd",
    "liability_score",
    "liability_num_violations",
]

structure_cols_high_good = [c for c in structure_cols_high_good if c in df.columns]
structure_cols_low_good = [c for c in structure_cols_low_good if c in df.columns]

def safe_z(series):
    series = pd.to_numeric(series, errors="coerce")
    if series.notna().sum() < 2 or series.std(ddof=0) == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.mean()) / series.std(ddof=0)

df["structure_score"] = 0.0

for col in structure_cols_high_good:
    df["structure_score"] += safe_z(df[col])

for col in structure_cols_low_good:
    df["structure_score"] -= safe_z(df[col])

df["model_score_z"] = safe_z(df[model_col])
df["tm_score_z"] = safe_z(df[tm_col])

if "mutation_positions" not in df.columns:
    df["mutation_positions"] = (
        df["mutations"]
        .fillna("")
        .str.findall(r"\d+")
        .apply(lambda xs: [int(x) for x in xs])
    )

all_positions = []

for positions in df["mutation_positions"]:
    if isinstance(positions, list):
        all_positions.extend(positions)

position_freq = pd.Series(all_positions).value_counts(normalize=True).to_dict()

def novelty_score(row):
    positions = row["mutation_positions"]

    if not isinstance(positions, list) or len(positions) == 0:
        return 0.0

    rarity = [1 - position_freq.get(pos, 0) for pos in positions]
    return np.mean(rarity) + 0.15 * row["n_mutations"]

df["novelty_score"] = df.apply(novelty_score, axis=1)
df["novelty_score_z"] = safe_z(df["novelty_score"])

df["lab_score"] = (
    0.50 * df["model_score_z"]
    + 0.20 * df["tm_score_z"]
    + 0.20 * df["structure_score"]
    + 0.10 * df["novelty_score_z"]
)

selected_ids = set()
# Select Group A: top activity model candidates
group_A = (
    df.sort_values(model_col, ascending=False)
    .head(10)
    .copy()
)
group_A["selection_group"] = "A_top_model"

selected_ids = set(group_A["variant_id"])

# Select Group B: unusual but structurally acceptable variants
group_B_hp = (
    df[
        (~df["variant_id"].isin(selected_ids))
        & (df["boltzgen_run"] == "HP_pocket")
        & (df["structure_score"] >= df["structure_score"].quantile(0.25))
    ]
    .sort_values("novelty_score", ascending=False)
    .head(3)
    .copy()
)

group_B_non_hp = (
    df[
        (~df["variant_id"].isin(selected_ids))
        & (df["boltzgen_run"] != "HP_pocket")
        & (df["structure_score"] >= df["structure_score"].quantile(0.25))
    ]
    .sort_values(["novelty_score", "structure_score"], ascending=False)
    .head(2)
    .copy()
)

group_B = pd.concat([group_B_hp, group_B_non_hp], ignore_index=True)
group_B["selection_group"] = "B_unusual_mutations"

selected_ids.update(group_B["variant_id"])

# Select Group C: structurally strong variants with medium model predictions
low_q = df[model_col].quantile(0.30)
high_q = df[model_col].quantile(0.70)

group_C = (
    df[
        (~df["variant_id"].isin(selected_ids))
        & (df[model_col] >= low_q)
        & (df[model_col] <= high_q)
    ]
    .sort_values("structure_score", ascending=False)
    .head(5)
    .copy()
)
group_C["selection_group"] = "C_good_structure_medium_model"

lab_top20 = pd.concat([group_A, group_B, group_C], ignore_index=True)

import re

def apply_mutations_to_reference(reference_seq, mutations):
    seq = list(reference_seq)

    if pd.isna(mutations) or str(mutations).strip() == "":
        return reference_seq

    mutation_list = [
        m.strip()
        for m in str(mutations).split(",")
        if m.strip() != ""
    ]

    for mut in mutation_list:
        match = re.fullmatch(r"([A-Z])(\d+)([A-Z])", mut)

        if match is None:
            raise ValueError(f"Could not parse mutation: {mut}")

        wt_aa, pos, new_aa = match.groups()
        pos = int(pos)

        current_aa = seq[pos - 1]

        if current_aa != wt_aa:
            raise ValueError(
                f"Reference mismatch for {mut}: "
                f"expected {wt_aa} at position {pos}, "
                f"but reference has {current_aa}"
            )

        seq[pos - 1] = new_aa

    return "".join(seq)


lab_top20["full_protein_sequence"] = lab_top20["mutations"].apply(
    lambda muts: apply_mutations_to_reference(reference_seq, muts)
)

lab_top20["sequence_length"] = lab_top20["full_protein_sequence"].str.len()

display_cols = [
    "selection_group",
    "variant_id",
    "boltzgen_run",
    "final_rank",
    model_col,
    tm_col,
    "lab_score",
    "structure_score",
    "novelty_score",
    "n_mutations",
    "mutations",
    "sequence_length",
    "full_protein_sequence",
    "design_to_target_iptm",
    "design_ptm",
    "filter_rmsd",
    "liability_score",
]

display_cols = [c for c in display_cols if c in lab_top20.columns]

lab_top20 = lab_top20[display_cols].sort_values(
    ["selection_group", model_col],
    ascending=[True, False]
)

print(lab_top20.to_string(index=False))
# Save the final lab test list as CSV
out_dir = project_root / "results" / "lab_variants"
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "lab_test_list_top20_boltzgen_with_sequences.csv"
lab_top20.to_csv(out_file, index=False)

print(f"\nSaved lab test list to:\n{out_file}")

fasta_file = out_dir / "lab_test_list_top20_boltzgen.fasta"

with open(fasta_file, "w") as f:
    for _, row in lab_top20.iterrows():
        safe_mutations = str(row["mutations"]).replace(",", "_").replace(" ", "")

        header = (
            f">{row['variant_id']}"
            f"|group={row['selection_group']}"
            f"|mutations={safe_mutations}"
        )

        f.write(header + "\n")

        seq = row["full_protein_sequence"]
        for i in range(0, len(seq), 80):
            f.write(seq[i:i + 80] + "\n")

print(f"Saved FASTA file to:\n{fasta_file}")

              selection_group                    variant_id boltzgen_run  final_rank  predicted_activity  predicted_tm  lab_score  structure_score  novelty_score  n_mutations                                          mutations  sequence_length                                                                                                                                                                                                                                                                                                                                               full_protein_sequence  design_to_target_iptm  design_ptm  filter_rmsd  liability_score
                  A_top_model       HP_pocket__HP_pocket_06    HP_pocket           7          284.587277     86.322468   2.177246         2.149502       1.922594            7            Y98S,D99R,G111Y,F134W,L137E,L139I,V144T              355 ANTTPVHALTDIDGGIAVDPAPRLAGPPVFGGPGNAAFDLAPVRSTGREMLRFDFPGVSIGAAHYEEGPTGATVIHIPAGARTAVDARGGAVG

# Detailed overview of what the code does

The workflow has five main stages.

First, the known experimental variants are converted into mutation-based features. The code reads the wild-type FASTA sequence and compares each known variant sequence against it. Each amino acid substitution is represented as a mutation string such as D304M. The code then calculates features such as mutation count, total charge change, hydrophobicity change, volume change, mutation presence at known positions, mutation presence at MD-contact positions, and one-hot encodings for mutation identities and positions. This is the feature table used for model training. The feature construction is shown in the pasted code where aa_properties, known_positions, md_contact_positions, extract_mutations, and build_sequence_feature_table are defined.

Second, the experimental labels are merged with the sequence features. The lab table is renamed so that variant ID, activity, and melting temperature have consistent column names. Activity and Tm values are converted into numeric values. The code then merges the sequence feature table with experimental PA6 activity and thermostability labels.

Third, two regression models are trained. Both models use a scikit-learn Pipeline with StandardScaler and ElasticNetCV. One model predicts PA6 activity and the other predicts melting temperature. Leave-one-out cross-validation is used, which is appropriate for a small dataset but still noisy. The reported performance is moderate, with activity LOOCV R² ≈ 0.45 and MAE ≈ 60, and Tm LOOCV R² ≈ 0.48 and MAE ≈ 1.38. That means the model is informative, but not reliable enough to use as the only selection criterion. The feature set and model performance are visible in the output file.

Fourth, the BoltzGen results are processed. The code reads all CSV files matching all_designs_metrics_*.csv, derives a run name from each filename, and concatenates all designs into one table. Because the BoltzGen sequences are not full-length WT-length sequences, the code uses local sequence alignment against the WT sequence. The alignment maps each designed chain onto the WT coordinate system and extracts mutations such as Y98S, D99R, F134W, or V144T. The resulting table contains all BoltzGen designs, their original BoltzGen scores, run metadata, alignment coordinates, and extracted mutations.

Fifth, the BoltzGen mutation lists are converted into the same model feature space as the known experimental variants. Missing one-hot columns are set to zero. This is necessary because the model was trained with a fixed set of columns. Predictions for activity and Tm are then generated and merged back into the BoltzGen table.

Finally, a balanced list of 20 lab candidates is selected. The final strategy is not simply “top 20 by predicted activity.” Instead, it uses three groups: Group A selects the 10 strongest predicted activity candidates, Group B selects unusual mutation patterns with acceptable structure quality, and Group C selects structurally strong variants with medium predicted activity. This is scientifically stronger because it tests both exploitation and exploration.

## Modules used

The visible workflow uses these Python modules:

pathlib.Path: Used for portable file and directory paths.

re: Used to extract residue numbers from mutation strings such as D304M.

pandas: Used for reading CSV files, constructing DataFrames, merging tables, generating feature matrices, filtering candidates, and exporting CSV outputs.

numpy: Used for numerical operations, especially in scoring and novelty calculations.

Bio.SeqIO: Used to read FASTA sequences.

Bio.Align.PairwiseAligner: Used for local sequence alignment between BoltzGen-designed partial sequences and the WT sequence.

sklearn.preprocessing.MultiLabelBinarizer: Used to one-hot encode mutation identities and mutated positions.

sklearn.pipeline.Pipeline: Used to combine feature scaling and regression into a reproducible modeling pipeline.

sklearn.preprocessing.StandardScaler: Used to standardize numeric features before ElasticNet regression.

sklearn.linear_model.ElasticNetCV: Used as the regression model for activity and Tm prediction.

sklearn.model_selection.LeaveOneOut: Used for leave-one-out cross-validation.

sklearn.model_selection.cross_val_predict: Used to generate cross-validated predictions.

sklearn.metrics.r2_score and mean_absolute_error: Used to evaluate model performance.

## Where heuristics are used

Several parts of the workflow are heuristic rather than strictly learned or experimentally validated.

The amino acid property table is a heuristic biochemical descriptor set. It uses manually assigned values for charge, hydrophobicity, and side-chain volume. These are useful but simplified. For example, histidine is assigned a partial charge of 0.5, which is a modeling choice and depends on pH and protonation state.

The sets known_positions = {99, 134, 301, 304, 330} and md_contact_positions = {91, 98, 111, 137, 139, 144, 146, 305} are heuristic feature groups. They encode prior biological assumptions: known positions come from previous experimental or literature-supported relevance, and MD-contact positions come from your structural/MD interpretation. This is reasonable, but it biases the model toward your current mechanistic hypothesis.

The local sequence alignment parameters are heuristic. You use a match score of 2, mismatch penalty of -1, gap opening penalty of -10, and gap extension penalty of -0.5. These values are reasonable for mapping close homologous protein fragments, but they are not uniquely determined.

Removing X residues before alignment is a heuristic cleaning step. It is practical because BoltzGen outputs unknown or masked residues, but it assumes that those X residues do not carry meaningful biological information.

Setting missing one-hot features to zero is a necessary modeling heuristic. For example, if a BoltzGen variant does not contain a mutation column that appeared during training, the code adds that feature as zero. This is correct for absent mutations, but it does not solve the deeper issue that many BoltzGen mutations were not seen during training.

The structure_score is heuristic. It combines z-scored BoltzGen structure metrics, adding metrics where higher is considered better and subtracting metrics where lower is considered better. The directionality is plausible, but the weights are not experimentally learned.

The novelty_score is heuristic. It rewards variants with rarer mutated positions and more mutations. This is useful for exploration, but it does not directly measure biochemical novelty or functional plausibility.

The final lab_score is also heuristic. It combines predicted activity, predicted Tm, structure score, and novelty score using chosen weights. This is not a trained model. It is a decision score for candidate selection.

The group selection strategy is heuristic but scientifically defensible. Group A maximizes expected activity, Group B maximizes exploration, and Group C tests whether structure-based scores identify promising candidates missed by the activity model.